# JSON5 Parser Testing

In [23]:
# Experimental setup
import random
import time
from statistics import mean, median

import json5
import coverage

from fuzzingbook.Fuzzer import RandomFuzzer
from fuzzingbook.GrammarCoverageFuzzer import TrackingGrammarCoverageFuzzer
from fuzzingbook.ProbabilisticGrammarFuzzer import ProbabilisticGrammarFuzzer, ExpansionCountMiner
from fuzzingbook.Parser import EarleyParser
from fuzzingbook.GrammarFuzzer import all_terminals
from fuzzingbook.Grammars import crange, extend_grammar, set_opts, opts
from fuzzingbook.MutationFuzzer import MutationFuzzer

SEED = 42
INPUT_COUNTS = [2500, 5000, 7500, 10000]
MAX_NONTERMINALS = 10
OBJECT_PROBABILITIES = [0.25, 0.40, 0.55]
MUTATION_SETTINGS = [(1, 3), (2, 5), (5, 10)]


In [24]:
# Random fuzzing
random_inputs = {}
generation_times = {}

for n in INPUT_COUNTS:
    random.seed(SEED)
    random_fuzzer = RandomFuzzer(min_length=1, max_length=100,
                                 char_start=32, char_range=95)

    start = time.perf_counter()
    random_inputs[n] = [random_fuzzer.fuzz() for _ in range(n)]
    generation_times['Random', n] = time.perf_counter() - start

    print("\nRANDOM FUZZER -", n, "INPUTS")
    print("Generated inputs:", len(random_inputs[n]))
    for i, text in enumerate(random_inputs[n][:5], 1):
        print(f"Input {i}: {text!r}")


RANDOM FUZZER - 2500 INPUTS
Generated inputs: 2500
Input 1: '.#~C?<1~-v~e+kV$#+;=`m#g9{syeU<YkC 4yVKC3;K-+P,MLmA%}Zd/P*fEpoNi8z(%t=E*=,PCZqN4OM'
Input 2: "uBywr)mq5d}?4[PBqxg<wI'=$HS"
Input 3: '(;h{H;s_RrZ2A1?gdAjVjSN<1a_+&.3p4wV'
Input 4: '(QPl[c@f!w|.wdBrK.EW4Z ||A`6`-pFq`m93O4ec lI^".NG>\'>h**}^(d00t\\f5AcmV;e}x9{GS'
Input 5: 'sOXbY/?<(K"kf=k< )zp\'=($J)a>Cu^;e0|ii\\?\\T8,,tWMVT[}&vsr,\'S}K-?88dY1V7C[?)Xf,&se!+>5T^]'

RANDOM FUZZER - 5000 INPUTS
Generated inputs: 5000
Input 1: '.#~C?<1~-v~e+kV$#+;=`m#g9{syeU<YkC 4yVKC3;K-+P,MLmA%}Zd/P*fEpoNi8z(%t=E*=,PCZqN4OM'
Input 2: "uBywr)mq5d}?4[PBqxg<wI'=$HS"
Input 3: '(;h{H;s_RrZ2A1?gdAjVjSN<1a_+&.3p4wV'
Input 4: '(QPl[c@f!w|.wdBrK.EW4Z ||A`6`-pFq`m93O4ec lI^".NG>\'>h**}^(d00t\\f5AcmV;e}x9{GS'
Input 5: 'sOXbY/?<(K"kf=k< )zp\'=($J)a>Cu^;e0|ii\\?\\T8,,tWMVT[}&vsr,\'S}K-?88dY1V7C[?)Xf,&se!+>5T^]'

RANDOM FUZZER - 7500 INPUTS
Generated inputs: 7500
Input 1: '.#~C?<1~-v~e+kV$#+;=`m#g9{syeU<YkC 4yVKC3;K-+P,MLmA%}Zd/P*fEpoNi8z(%t=E*=,PCZqN4OM'
Inpu

In [25]:
code_coverage = {}
parser_results = {}

# Random parser code coverage
for n, inputs in random_inputs.items():
    cov = coverage.Coverage(source=["json5"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5.loads(test_input)
                accepted += 1
            except ValueError:
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nRANDOM FUZZER - CODE COVERAGE")
    print("Input count:", n)
    code_coverage['Random', n] = cov.report()
    parser_results['Random', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage['Random', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")


RANDOM FUZZER - CODE COVERAGE
Input count: 2500
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    271    11%
.venv\Lib\site-packages\json5\parser.py       632    231    63%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    601    42%

Total line coverage: 41.88%
Accepted: 2
Rejected: 2498
Unexpected errors: 0
Acceptance rate: 0.08%

RANDOM FUZZER - CODE COVERAGE
Input count: 5000
Name                                        Stmts   Miss  Cover
----------------------------------

In [26]:
# JSON5 grammar construction
JSON5_GRAMMAR = {
    "<start>": ["<value>"],

    "<value>": [
        "<object>", "<array>", "<string>", "<number>",
        "true", "false", "null"
    ],

    "<object>": [
        "{}", "{<members>}", "{<members>,}",
        "{ <members> }", "{<space><members><space>}",
        "{//comment\n<members>}"
    ],
    "<members>": [
        "<pair>", "<pair>, <members>",
        "<pair>,<space><members>"
    ],
    "<pair>": ["<key>: <value>", "<key>:<value>"],
    "<key>": ["<identifier>", "<string>"],

    "<identifier>": [
        "<id_start>", "<id_start><id_part>",
        "<id_start><id_part><id_part>",
        "<id_start><id_part><id_part><id_part>"
    ],
    "<id_start>": crange("a", "z") + crange("A", "Z") + ["_", "$"],
    "<id_part>": crange("a", "z") + crange("A", "Z") + crange("0", "9") + ["_", "$"],

    "<array>": [
        "[]", "[<elements>]", "[<elements>,]",
        "[ <elements> ]", "[<space><elements><space>]"
    ],
    "<elements>": [
        "<value>", "<value>, <elements>",
        "<value>,<space><elements>"
    ],

    "<string>": [
        '""', "''", '"<chars>"', "'<chars>'",
        '"<chars>\\n<chars>"', "'<chars>\\t<chars>'"
    ],
    "<chars>": [
        "<char>", "<char><char>", "<char><char><char>",
        "<char><char><char><char>"
    ],
    "<char>": crange("a", "z") + crange("A", "Z") + crange("0", "9") + [" ", "_", "-", "."],

    "<number>": [
        "<integer>", "<integer>.<digits>",
        "<integer>e<exp_sign><digits>", "0x<hex_digits>",
        "+<unsigned_integer>", "<integer>.",
        ".<digits>", "-.<digits>",
        "Infinity", "-Infinity", "NaN"
    ],
    "<integer>": ["0", "<unsigned_integer>", "-<unsigned_integer>"],
    "<unsigned_integer>": [
        "<nonzero>", "<nonzero><digit>",
        "<nonzero><digit><digit>"
    ],
    "<nonzero>": crange("1", "9"),
    "<digit>": crange("0", "9"),
    "<digits>": ["<digit>", "<digit><digit>", "<digit><digit><digit>"],
    "<hex_digits>": ["<hex_digit>", "<hex_digit><hex_digit>", "<hex_digit><hex_digit><hex_digit>"],
    "<hex_digit>": crange("0", "9") + crange("a", "f") + crange("A", "F"),
    "<exp_sign>": ["", "+", "-"],
    "<space>": [" ", "\n", "\t"]
}

print('Grammar non-terminals:', len(JSON5_GRAMMAR))

Grammar non-terminals: 24


In [27]:
# Normal grammar fuzzing
normal_inputs = {}
normal_used = {}
all_productions = TrackingGrammarCoverageFuzzer(JSON5_GRAMMAR).max_expansion_coverage()

for n in INPUT_COUNTS:
    random.seed(SEED)
    normal_fuzzer = TrackingGrammarCoverageFuzzer(
        JSON5_GRAMMAR, max_nonterminals=MAX_NONTERMINALS
    )

    start = time.perf_counter()
    normal_inputs[n] = [normal_fuzzer.fuzz() for _ in range(n)]
    generation_times['Normal', n] = time.perf_counter() - start
    normal_used[n] = set(normal_fuzzer.expansion_coverage())

    print("\nNORMAL GRAMMAR FUZZER -", n, "INPUTS")
    print("Generated inputs:", len(normal_inputs[n]))
    for i, text in enumerate(normal_inputs[n][:5], 1):
        print(f"Input {i}: {text}")


NORMAL GRAMMAR FUZZER - 2500 INPUTS
Generated inputs: 2500
Input 1: false
Input 2: [ false, true ]
Input 3: {	"": true
}
Input 4: false
Input 5: false

NORMAL GRAMMAR FUZZER - 5000 INPUTS
Generated inputs: 5000
Input 1: false
Input 2: [ false, true ]
Input 3: {	"": true
}
Input 4: false
Input 5: false

NORMAL GRAMMAR FUZZER - 7500 INPUTS
Generated inputs: 7500
Input 1: false
Input 2: [ false, true ]
Input 3: {	"": true
}
Input 4: false
Input 5: false

NORMAL GRAMMAR FUZZER - 10000 INPUTS
Generated inputs: 10000
Input 1: false
Input 2: [ false, true ]
Input 3: {	"": true
}
Input 4: false
Input 5: false


In [28]:
# Normal generated input testing
for n, inputs in normal_inputs.items():
    accepted = 0
    rejected = 0
    unexpected = 0

    print("\nNORMAL GENERATED INPUT TESTING -", n, "INPUTS")
    for i, text in enumerate(inputs, 1):
        try:
            json5.loads(text)
            accepted += 1
            status = "ACCEPTED"
        except ValueError:
            rejected += 1
            status = "REJECTED"
        except Exception as error:
            unexpected += 1
            status = "UNEXPECTED ERROR"
            print(type(error).__name__, error)

        if i <= 5:
            print(f"Test {i}: {status}")
            print(f"Input: {text}")

    print("\n--- TEST SUMMARY ---")
    print("Total inputs:", n)
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")


NORMAL GENERATED INPUT TESTING - 2500 INPUTS
Test 1: ACCEPTED
Input: false
Test 2: ACCEPTED
Input: [ false, true ]
Test 3: ACCEPTED
Input: {	"": true
}
Test 4: ACCEPTED
Input: false
Test 5: ACCEPTED
Input: false

--- TEST SUMMARY ---
Total inputs: 2500
Accepted: 2500
Rejected: 0
Unexpected errors: 0
Acceptance rate: 100.00%

NORMAL GENERATED INPUT TESTING - 5000 INPUTS
Test 1: ACCEPTED
Input: false
Test 2: ACCEPTED
Input: [ false, true ]
Test 3: ACCEPTED
Input: {	"": true
}
Test 4: ACCEPTED
Input: false
Test 5: ACCEPTED
Input: false

--- TEST SUMMARY ---
Total inputs: 5000
Accepted: 5000
Rejected: 0
Unexpected errors: 0
Acceptance rate: 100.00%

NORMAL GENERATED INPUT TESTING - 7500 INPUTS
Test 1: ACCEPTED
Input: false
Test 2: ACCEPTED
Input: [ false, true ]
Test 3: ACCEPTED
Input: {	"": true
}
Test 4: ACCEPTED
Input: false
Test 5: ACCEPTED
Input: false

--- TEST SUMMARY ---
Total inputs: 7500
Accepted: 7500
Rejected: 0
Unexpected errors: 0
Acceptance rate: 100.00%

NORMAL GENERATED I

In [29]:
# Normal grammar production coverage
normal_grammar_coverage = {}
for n in INPUT_COUNTS:
    used = normal_used[n]
    normal_grammar_coverage[n] = len(used) / len(all_productions) * 100

    print("\nNORMAL GRAMMAR COVERAGE -", n, "INPUTS")
    print("Total grammar productions:", len(all_productions))
    print("Used productions:", len(used))
    print(f"Grammar coverage: {normal_grammar_coverage[n]:.2f}%")
    print("Missing productions:", sorted(all_productions - used))


NORMAL GRAMMAR COVERAGE - 2500 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

NORMAL GRAMMAR COVERAGE - 5000 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

NORMAL GRAMMAR COVERAGE - 7500 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

NORMAL GRAMMAR COVERAGE - 10000 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []


In [30]:
# Normal parser code coverage
for n, inputs in normal_inputs.items():
    cov = coverage.Coverage(source=["json5"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5.loads(test_input)
                accepted += 1
            except ValueError:
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nNORMAL FUZZER - CODE COVERAGE")
    print("Input count:", n)
    code_coverage['Normal', n] = cov.report()
    parser_results['Normal', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage['Normal', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")


NORMAL FUZZER - CODE COVERAGE
Input count: 2500
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    253    17%
.venv\Lib\site-packages\json5\parser.py       632    267    58%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    619    40%

Total line coverage: 40.14%
Accepted: 2500
Rejected: 0
Unexpected errors: 0
Acceptance rate: 100.00%

NORMAL FUZZER - CODE COVERAGE
Input count: 5000
Name                                        Stmts   Miss  Cover
--------------------------------

In [31]:
# Heuristic-guided grammar fuzzer
heuristic_grammars = {}
for probability in OBJECT_PROBABILITIES:
    guided_grammar = extend_grammar(JSON5_GRAMMAR)
    set_opts(guided_grammar, '<value>', '<object>', opts(prob=probability))
    heuristic_grammars[probability] = guided_grammar
    print("Object probability:", probability)

Object probability: 0.25
Object probability: 0.4
Object probability: 0.55


In [32]:
# Heuristic input generation
heuristic_inputs = {}
heuristic_used = {}

for probability in OBJECT_PROBABILITIES:
    for n in INPUT_COUNTS:
        random.seed(SEED)
        guided_fuzzer = ProbabilisticGrammarFuzzer(
            heuristic_grammars[probability], max_nonterminals=MAX_NONTERMINALS
        )
        grammar_miner = ExpansionCountMiner(EarleyParser(JSON5_GRAMMAR))

        inputs = []
        generation_time = 0
        for _ in range(n):
            start = time.perf_counter()
            tree = guided_fuzzer.fuzz_tree()
            text = all_terminals(tree)
            generation_time += time.perf_counter() - start
            inputs.append(text)
            grammar_miner.add_tree(tree)

        heuristic_inputs[probability, n] = inputs
        heuristic_used[probability, n] = set(grammar_miner.counts())
        generation_times[f'Object {probability:.0%}', n] = generation_time

        print(f"\nHEURISTIC FUZZER - OBJECT {probability:.0%} - {n} INPUTS")
        print("Generated inputs:", len(inputs))
        for i, text in enumerate(inputs[:5], 1):
            print(f"Input {i}: {text}")


HEURISTIC FUZZER - OBJECT 25% - 2500 INPUTS
Generated inputs: 2500
Input 1: true
Input 2: true
Input 3: -85.72
Input 4: [false, null, {	d: Infinity, q: NaN, $YQS:{//comment
'': {'': true, "":null}}, "":"D"	}]
Input 5: {FVm:null, 'SL\twaxD':-.682, O5PG:true,}

HEURISTIC FUZZER - OBJECT 25% - 5000 INPUTS
Generated inputs: 5000
Input 1: true
Input 2: true
Input 3: -85.72
Input 4: [false, null, {	d: Infinity, q: NaN, $YQS:{//comment
'': {'': true, "":null}}, "":"D"	}]
Input 5: {FVm:null, 'SL\twaxD':-.682, O5PG:true,}

HEURISTIC FUZZER - OBJECT 25% - 7500 INPUTS
Generated inputs: 7500
Input 1: true
Input 2: true
Input 3: -85.72
Input 4: [false, null, {	d: Infinity, q: NaN, $YQS:{//comment
'': {'': true, "":null}}, "":"D"	}]
Input 5: {FVm:null, 'SL\twaxD':-.682, O5PG:true,}

HEURISTIC FUZZER - OBJECT 25% - 10000 INPUTS
Generated inputs: 10000
Input 1: true
Input 2: true
Input 3: -85.72
Input 4: [false, null, {	d: Infinity, q: NaN, $YQS:{//comment
'': {'': true, "":null}}, "":"D"	}]
Input 5:

In [33]:
# Heuristic grammar production coverage
heuristic_grammar_coverage = {}
for (probability, n), used in heuristic_used.items():
    heuristic_grammar_coverage[probability, n] = len(used & all_productions) / len(all_productions) * 100

    print(f"\nHEURISTIC GRAMMAR COVERAGE - OBJECT {probability:.0%} - {n} INPUTS")
    print("Total grammar productions:", len(all_productions))
    print("Used productions:", len(used & all_productions))
    print(f"Grammar coverage: {heuristic_grammar_coverage[probability, n]:.2f}%")
    print("Missing productions:", sorted(all_productions - used))


HEURISTIC GRAMMAR COVERAGE - OBJECT 25% - 2500 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 25% - 5000 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 25% - 7500 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 25% - 10000 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 40% - 2500 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 40% - 5000 INPUTS
Total grammar productions: 297
Used productions: 297
Grammar coverage: 100.00%
Missing productions: []

HEURISTIC GRAMMAR COVERAGE - OBJECT 40% - 7500 INP

In [34]:
# Heuristic parser code coverage
for (probability, n), inputs in heuristic_inputs.items():
    cov = coverage.Coverage(source=["json5"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5.loads(test_input)
                accepted += 1
            except ValueError:
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nHEURISTIC FUZZER - CODE COVERAGE")
    print(f"Object probability: {probability:.0%}")
    print("Input count:", n)
    code_coverage[f'Object {probability:.0%}', n] = cov.report()
    parser_results[f'Object {probability:.0%}', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage[f'Object {probability:.0%}', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")


HEURISTIC FUZZER - CODE COVERAGE
Object probability: 25%
Input count: 2500
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    253    17%
.venv\Lib\site-packages\json5\parser.py       632    267    58%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    619    40%

Total line coverage: 40.14%
Accepted: 2500
Rejected: 0
Unexpected errors: 0
Acceptance rate: 100.00%

HEURISTIC FUZZER - CODE COVERAGE
Object probability: 25%
Input count: 5000
Name                                      

In [35]:
# Input complexity analysis
complexity_inputs = {}
for n, inputs in normal_inputs.items():
    complexity_inputs['Normal', n] = inputs
for (probability, n), inputs in heuristic_inputs.items():
    complexity_inputs[f'Object {probability:.0%}', n] = inputs

for (label, n), inputs in complexity_inputs.items():
    depths = []
    lengths = []
    for text in inputs:
        depth = 0
        largest = 0
        for char in text:
            if char == '{':
                depth += 1
                largest = max(largest, depth)
            elif char == '}':
                depth -= 1
        depths.append(largest)
        lengths.append(len(text))

    print(f"\n{label} - {n} INPUTS")
    print(f"Average object depth: {mean(depths):.2f}")
    print("Maximum object depth:", max(depths))
    print(f"Average input length: {mean(lengths):.2f}")


Normal - 2500 INPUTS
Average object depth: 0.23
Maximum object depth: 3
Average input length: 12.24

Normal - 5000 INPUTS
Average object depth: 0.23
Maximum object depth: 4
Average input length: 12.17

Normal - 7500 INPUTS
Average object depth: 0.24
Maximum object depth: 4
Average input length: 12.50

Normal - 10000 INPUTS
Average object depth: 0.24
Maximum object depth: 4
Average input length: 12.69

Object 25% - 2500 INPUTS
Average object depth: 0.43
Maximum object depth: 4
Average input length: 16.35

Object 25% - 5000 INPUTS
Average object depth: 0.44
Maximum object depth: 5
Average input length: 16.58

Object 25% - 7500 INPUTS
Average object depth: 0.46
Maximum object depth: 5
Average input length: 16.90

Object 25% - 10000 INPUTS
Average object depth: 0.46
Maximum object depth: 5
Average input length: 17.01

Object 40% - 2500 INPUTS
Average object depth: 0.78
Maximum object depth: 9
Average input length: 23.68

Object 40% - 5000 INPUTS
Average object depth: 0.80
Maximum object d

In [36]:
# Mutation-based fuzzing
seed_inputs = [text for text in normal_inputs[INPUT_COUNTS[0]] if len(text) > 10][:20]
assert len(seed_inputs) == 20
mutation_inputs = {}

for minimum, maximum in MUTATION_SETTINGS:
    for n in INPUT_COUNTS:
        random.seed(SEED)
        mutation_fuzzer = MutationFuzzer(
            seed=seed_inputs, min_mutations=minimum, max_mutations=maximum
        )

        for _ in seed_inputs:
            mutation_fuzzer.fuzz()

        start = time.perf_counter()
        inputs = [mutation_fuzzer.fuzz() for _ in range(n)]
        generation_times[f'Mutation {minimum}-{maximum}', n] = time.perf_counter() - start
        mutation_inputs[minimum, maximum, n] = inputs

        print(f"\nMUTATION FUZZER - {minimum}-{maximum} - {n} INPUTS")
        print("Generated inputs:", len(inputs))
        for i, text in enumerate(inputs[:5], 1):
            print(f"Input {i}: {text!r}")


MUTATION FUZZER - 1-3 - 2500 INPUTS
Generated inputs: 2500
Input 1: '{\tH3: "W,-",\t"T0e":false }'
Input 2: '{Ge2n:Null, "zvK_":"", \'\':true,\t\'\': null, "": true,\t"": false,}'
Input 3: '[ [trud,]\t]'
Input 4: '[ flse, true ]'
Input 5: "'4BYG\\t9W'"

MUTATION FUZZER - 1-3 - 5000 INPUTS
Generated inputs: 5000
Input 1: '{\tH3: "W,-",\t"T0e":false }'
Input 2: '{Ge2n:Null, "zvK_":"", \'\':true,\t\'\': null, "": true,\t"": false,}'
Input 3: '[ [trud,]\t]'
Input 4: '[ flse, true ]'
Input 5: "'4BYG\\t9W'"

MUTATION FUZZER - 1-3 - 7500 INPUTS
Generated inputs: 7500
Input 1: '{\tH3: "W,-",\t"T0e":false }'
Input 2: '{Ge2n:Null, "zvK_":"", \'\':true,\t\'\': null, "": true,\t"": false,}'
Input 3: '[ [trud,]\t]'
Input 4: '[ flse, true ]'
Input 5: "'4BYG\\t9W'"

MUTATION FUZZER - 1-3 - 10000 INPUTS
Generated inputs: 10000
Input 1: '{\tH3: "W,-",\t"T0e":false }'
Input 2: '{Ge2n:Null, "zvK_":"", \'\':true,\t\'\': null, "": true,\t"": false,}'
Input 3: '[ [trud,]\t]'
Input 4: '[ flse, true ]'
Input

In [37]:
# Mutation testing and parser code coverage
for (minimum, maximum, n), inputs in mutation_inputs.items():
    cov = coverage.Coverage(source=["json5"], branch=False, data_file=None)
    accepted = 0
    rejected = 0
    unexpected = 0

    cov.start()
    try:
        for test_input in inputs:
            try:
                json5.loads(test_input)
                accepted += 1
            except ValueError:
                rejected += 1
            except Exception as error:
                unexpected += 1
                print(type(error).__name__, error)
    finally:
        cov.stop()

    print("\nMUTATION FUZZER - CODE COVERAGE")
    print(f"Mutations: {minimum}-{maximum}")
    print("Input count:", n)
    code_coverage[f'Mutation {minimum}-{maximum}', n] = cov.report()
    parser_results[f'Mutation {minimum}-{maximum}', n] = (accepted, rejected, unexpected)

    print(f"\nTotal line coverage: {code_coverage[f'Mutation {minimum}-{maximum}', n]:.2f}%")
    print("Accepted:", accepted)
    print("Rejected:", rejected)
    print("Unexpected errors:", unexpected)
    print(f"Acceptance rate: {accepted / n * 100:.2f}%")


MUTATION FUZZER - CODE COVERAGE
Mutations: 1-3
Input count: 2500
Name                                        Stmts   Miss  Cover
---------------------------------------------------------------
.venv\Lib\site-packages\json5\__init__.py       3      3     0%
.venv\Lib\site-packages\json5\__main__.py       4      4     0%
.venv\Lib\site-packages\json5\host.py          29     29     0%
.venv\Lib\site-packages\json5\lib.py          303    252    17%
.venv\Lib\site-packages\json5\parser.py       632    233    63%
.venv\Lib\site-packages\json5\tool.py          61     61     0%
.venv\Lib\site-packages\json5\version.py        2      2     0%
---------------------------------------------------------------
TOTAL                                        1034    584    44%

Total line coverage: 43.52%
Accepted: 449
Rejected: 2051
Unexpected errors: 0
Acceptance rate: 17.96%

MUTATION FUZZER - CODE COVERAGE
Mutations: 1-3
Input count: 5000
Name                                        Stmts   Miss  Cov

In [38]:
# Parser execution time
input_groups = {}
for n, inputs in random_inputs.items():
    input_groups['Random', n] = inputs
for n, inputs in normal_inputs.items():
    input_groups['Normal', n] = inputs
for (probability, n), inputs in heuristic_inputs.items():
    input_groups[f'Object {probability:.0%}', n] = inputs
for (minimum, maximum, n), inputs in mutation_inputs.items():
    input_groups[f'Mutation {minimum}-{maximum}', n] = inputs

parse_times = {}
for (label, n), inputs in input_groups.items():
    times = []
    for _ in range(3):
        start = time.perf_counter()
        for text in inputs:
            try:
                json5.loads(text)
            except Exception:
                pass
        times.append(time.perf_counter() - start)

    parse_times[label, n] = median(times)
    print(f"{label} - {n} inputs: {parse_times[label, n]:.4f} seconds")

Random - 2500 inputs: 0.2826 seconds
Random - 5000 inputs: 0.5685 seconds
Random - 7500 inputs: 0.9709 seconds
Random - 10000 inputs: 1.4590 seconds
Normal - 2500 inputs: 0.6319 seconds
Normal - 5000 inputs: 1.2923 seconds
Normal - 7500 inputs: 2.0046 seconds
Normal - 10000 inputs: 2.1371 seconds
Object 25% - 2500 inputs: 0.7022 seconds
Object 25% - 5000 inputs: 1.3990 seconds
Object 25% - 7500 inputs: 2.3699 seconds
Object 25% - 10000 inputs: 3.2522 seconds
Object 40% - 2500 inputs: 1.0695 seconds
Object 40% - 5000 inputs: 2.0941 seconds
Object 40% - 7500 inputs: 3.0805 seconds
Object 40% - 10000 inputs: 4.6159 seconds
Object 55% - 2500 inputs: 1.4100 seconds
Object 55% - 5000 inputs: 3.3349 seconds
Object 55% - 7500 inputs: 5.6754 seconds
Object 55% - 10000 inputs: 7.0065 seconds
Mutation 1-3 - 2500 inputs: 0.9547 seconds
Mutation 1-3 - 5000 inputs: 2.0833 seconds
Mutation 1-3 - 7500 inputs: 2.9308 seconds
Mutation 1-3 - 10000 inputs: 3.5922 seconds
Mutation 2-5 - 2500 inputs: 0.7285

In [39]:
# Input generation time
# Normal generation includes grammar tracking.
for (label, n), seconds in generation_times.items():
    print(f"{label} - {n} inputs: {seconds:.4f} seconds")

Random - 2500 inputs: 0.0878 seconds
Random - 5000 inputs: 0.1372 seconds
Random - 7500 inputs: 0.2057 seconds
Random - 10000 inputs: 0.3222 seconds
Normal - 2500 inputs: 17.3899 seconds
Normal - 5000 inputs: 32.7994 seconds
Normal - 7500 inputs: 45.3861 seconds
Normal - 10000 inputs: 66.2189 seconds
Object 25% - 2500 inputs: 36.2666 seconds
Object 25% - 5000 inputs: 86.4144 seconds
Object 25% - 7500 inputs: 102.3009 seconds
Object 25% - 10000 inputs: 130.7996 seconds
Object 40% - 2500 inputs: 81.2678 seconds
Object 40% - 5000 inputs: 148.1636 seconds
Object 40% - 7500 inputs: 248.9715 seconds
Object 40% - 10000 inputs: 448.1905 seconds
Object 55% - 2500 inputs: 221.8899 seconds
Object 55% - 5000 inputs: 280.4959 seconds
Object 55% - 7500 inputs: 419.8077 seconds
Object 55% - 10000 inputs: 516.5151 seconds
Mutation 1-3 - 2500 inputs: 0.0296 seconds
Mutation 1-3 - 5000 inputs: 0.0511 seconds
Mutation 1-3 - 7500 inputs: 0.0880 seconds
Mutation 1-3 - 10000 inputs: 0.0766 seconds
Mutation 

In [40]:
# Final experimental results
for n in INPUT_COUNTS:
    print(f"\nFINAL RESULTS - {n} INPUTS")
    print("-" * 40)

    print("\nCODE COVERAGE")
    for (label, count), percentage in code_coverage.items():
        if count == n:
            print(f"{label}: {percentage:.2f}%")

    print("\nGRAMMAR PRODUCTION COVERAGE")
    print(f"Normal: {normal_grammar_coverage[n]:.2f}%")
    for probability in OBJECT_PROBABILITIES:
        print(f"Object {probability:.0%}: {heuristic_grammar_coverage[probability, n]:.2f}%")

    print("\nACCEPTANCE")
    for (label, count), (accepted, rejected, unexpected) in parser_results.items():
        if count == n:
            print(f"{label}: {accepted / n * 100:.2f}% | Accepted: {accepted} | Rejected: {rejected} | Unexpected: {unexpected}")

    print("\nTOTAL TESTING TIME")
    for (label, count), seconds in generation_times.items():
        if count == n:
            total = seconds + parse_times[label, n]
            print(f"{label}: {total:.4f} seconds")


FINAL RESULTS - 2500 INPUTS
----------------------------------------

CODE COVERAGE
Random: 41.88%
Normal: 40.14%
Object 25%: 40.14%
Object 40%: 40.14%
Object 55%: 40.14%
Mutation 1-3: 43.52%
Mutation 2-5: 43.71%
Mutation 5-10: 42.75%

GRAMMAR PRODUCTION COVERAGE
Normal: 100.00%
Object 25%: 100.00%
Object 40%: 100.00%
Object 55%: 100.00%

ACCEPTANCE
Random: 0.08% | Accepted: 2 | Rejected: 2498 | Unexpected: 0
Normal: 100.00% | Accepted: 2500 | Rejected: 0 | Unexpected: 0
Object 25%: 100.00% | Accepted: 2500 | Rejected: 0 | Unexpected: 0
Object 40%: 100.00% | Accepted: 2500 | Rejected: 0 | Unexpected: 0
Object 55%: 100.00% | Accepted: 2500 | Rejected: 0 | Unexpected: 0
Mutation 1-3: 17.96% | Accepted: 449 | Rejected: 2051 | Unexpected: 0
Mutation 2-5: 9.28% | Accepted: 232 | Rejected: 2268 | Unexpected: 0
Mutation 5-10: 3.64% | Accepted: 91 | Rejected: 2409 | Unexpected: 0

TOTAL TESTING TIME
Random: 0.3704 seconds
Normal: 18.0219 seconds
Object 25%: 36.9689 seconds
Object 40%: 82.3373